In [1]:

import pandas as pd
# allows us to use pandas for reading, cleaning, and exporting our dataset

import numpy as np
# allows us to use numpy for any numeric operations we need during cleaning

In [2]:
financial_fraud = pd.read_csv('../data/PS_20174392719_1491204439457_log.csv')
# reads the raw dataset back in fresh, so this notebook starts clean and independent from the EDA notebook

In [3]:
financial_fraud.isnull().sum()
# double checks for null values in this fresh copy before we do anything else, confirming what we found in EDA still holds

step              0
type              0
amount            0
nameOrig          0
oldbalanceOrg     0
newbalanceOrig    0
nameDest          0
oldbalanceDest    0
newbalanceDest    0
isFraud           0
isFlaggedFraud    0
dtype: int64

In [ ]:
financial_fraud = financial_fraud.drop(columns=['nameOrig', 'nameDest'])
# drops the two high cardinality ID columns; they're unique identifiers, not predictive features, and would only add noise or blow up dimensionality if encoded

In [5]:
financial_fraud['orig_balance_delta'] = financial_fraud['oldbalanceOrg'] - financial_fraud['newbalanceOrig']
# engineers a new feature capturing how much money actually left the origin account, based on our bivariate finding that this delta is elevated for fraud

In [ ]:
financial_fraud['is_zero_amt_cashout'] = ((financial_fraud['amount'] == 0) & (financial_fraud['type'] == 'CASH_OUT')).astype(int)
# engineers a flag for the rare but extremely high confidence pattern found in EDA: zero amount CASH_OUT transactions had a 100% fraud rate
# .astype(int) converts the True/False result into 1/0 so it's usable directly as a numeric model feature

In [7]:
financial_fraud = pd.get_dummies(financial_fraud, columns=['type'], drop_first=True)
# converts the categorical type column into numeric dummy/indicator columns since models require numeric input
# drop_first=True avoids redundant columns (multicollinearity) by dropping one category as the baseline

In [8]:
financial_fraud.head()
# visually confirms the new columns were added correctly and the dropped/encoded columns are no longer present in their original form

,step,amount,oldbalanceOrg,newbalanceOrig,oldbalanceDest,newbalanceDest,isFraud,isFlaggedFraud,orig_balance_delta,is_zero_amt_cashout,type_CASH_OUT,type_DEBIT,type_PAYMENT,type_TRANSFER
0,1,9839.64,170136.0,160296.36,0.0,0.0,0,0,9839.64,0,False,False,True,False
1,1,1864.28,21249.0,19384.72,0.0,0.0,0,0,1864.28,0,False,False,True,False
2,1,181.00,181.0,0.00,0.0,0.0,1,0,181.00,0,False,False,False,True
3,1,181.00,181.0,0.00,21182.0,0.0,1,0,181.00,0,True,False,False,False
4,1,11668.14,41554.0,29885.86,0.0,0.0,0,0,11668.14,0,False,False,True,False


In [9]:
financial_fraud.info()
# confirms there are no unexpected nulls introduced by the new features and that dtypes look correct (numeric, no leftover object columns except what we intended)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6362620 entries, 0 to 6362619
Data columns (total 14 columns):
 #   Column               Dtype  
---  ------               -----  
 0   step                 int64  
 1   amount               float64
 2   oldbalanceOrg        float64
 3   newbalanceOrig       float64
 4   oldbalanceDest       float64
 5   newbalanceDest       float64
 6   isFraud              int64  
 7   isFlaggedFraud       int64  
 8   orig_balance_delta   float64
 9   is_zero_amt_cashout  int64  
 10  type_CASH_OUT        bool   
 11  type_DEBIT           bool   
 12  type_PAYMENT         bool   
 13  type_TRANSFER        bool   
dtypes: bool(4), float64(6), int64(4)
memory usage: 509.7 MB


Cleaning summary: Dropped nameOrig/nameDest (non predictive IDs). No outlier removal, outliers represent real fraud signal in this dataset. Engineered orig_balance_delta and is_zero_amt_cashout based on EDA findings. Encoded type via one hot encoding. Confirmed no nulls and correct dtypes before export.

In [ ]:
bool_cols = financial_fraud.select_dtypes(include='bool').columns
# selects only the columns with a boolean dtype (the type_ dummy columns created by pd.get_dummies)
# .columns grabs just their names, so we know exactly which columns to convert
financial_fraud[bool_cols] = financial_fraud[bool_cols].astype(int)
# converts those boolean columns (True/False) into integers (1/0)
# this avoids any dtype issues when this data is loaded into scikit-learn for modeling later

In [11]:
financial_fraud.to_csv('../data/cleaned_fraud.csv', index=False)
# exports the cleaned, feature engineered dataframe to a new CSV file for use in the modeling notebook
# index=False prevents pandas from writing the row index as an extra unwanted column